# 🔬 Comprehensive Hyperparameter Tuning\n\n**Dataset**: CoNLL-2003\n**Experiments**: 8 systematic variations\n\n1. Baseline (ReLU)\n2. Baseline (GELU)\n3. More Receptors (256)\n4. More Glomeruli (64)\n5. Larger LSTM (512)\n6. Lower Dropout (0.2)\n7. Larger Batch (64)\n8. Strong Regularization\n\n**Each experiment auto**:\n- Trains & evaluates\n- Saves model → Google Drive\n- Saves results → GitHub\n- Generates curves & plots

## Setup

In [ ]:
import torch\nprint(f'PyTorch: {torch.__version__}')\nprint(f'CUDA: {torch.cuda.is_available()}')\nif torch.cuda.is_available():\n    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
!git clone https://github.com/bhushan1729/olfaction-inspired-ner.git\n%cd olfaction-inspired-ner

In [ ]:
!pip install -q torch numpy scikit-learn seqeval matplotlib seaborn pandas tqdm tensorboard

In [ ]:
import os\nif not os.path.exists('./data/glove.6B.300d.txt'):\n    !mkdir -p data\n    !wget -q http://nlp.stanford.edu/data/glove.6B.zip -O data/glove.6B.zip\n    !unzip -q data/glove.6B.zip -d data/\n    !rm data/glove.6B.zip

## Google Drive

In [ ]:
from google.colab import drive\ndrive.mount('/content/drive')\n\nmodel_dir = '/content/drive/MyDrive/olfaction_ner/models'\n!mkdir -p {model_dir}

## Helpers

In [ ]:
import json, shutil, matplotlib.pyplot as plt\nfrom pathlib import Path\n\n!mkdir -p comparison_plots experiment_results/CoNLL-2003\nall_experiments = []\n\ndef save_to_drive(name, save_dir):\n    dst = f'/content/drive/MyDrive/olfaction_ner/models/{name}.pt'\n    shutil.copy(f'{save_dir}/best_model.pt', dst)\n    print(f'✓ Drive: {dst}')\n\ndef save_results(name, cfg, res, sd):\n    ed = f'experiment_results/CoNLL-2003/{name}'\n    Path(ed).mkdir(parents=True, exist_ok=True)\n    with open(f'{ed}/metadata.json', 'w') as f:\n        json.dump({'name': name, 'config': cfg}, f, indent=2)\n    with open(f'{ed}/results.json', 'w') as f:\n        json.dump(res, f, indent=2)\n    all_experiments.append({'name': name, 'config': cfg, 'results': res})\n\ndef plot_curves(name, res, path):\n    fig, ax = plt.subplots(1, 2, figsize=(14, 5))\n    epochs = [e['epoch'] for e in res['epochs']]\n    ax[0].plot(epochs, [e['train']['total_loss'] for e in res['epochs']])\n    ax[0].set_title(f'{name}: Loss')\n    ax[1].plot(epochs, [e['valid']['f1'] for e in res['epochs']], color='green')\n    ax[1].set_title(f'{name}: F1')\n    plt.tight_layout()\n    plt.savefig(path, dpi=150, bbox_inches='tight')\n    plt.close()\nprint('✓ Ready')

---\n# Experiments\n---

## Experiment 1: Baseline (ReLU)

In [ ]:
!python src/train.py --config config/experiments.yaml --experiment olfactory_full --save_dir results/tuning/exp1_relu

In [ ]:
with open('results/tuning/exp1_relu/results.json') as f:\n    results = json.load(f)\n\ncfg = {'activation':'relu','receptors':128,'glomeruli':32,'lstm':256,'dropout':0.5,'batch':32,'sparse':0.001,'diverse':0.01}\nsave_to_drive('exp1_relu', 'results/tuning/exp1_relu')\nsave_results('exp1_relu', cfg, results, 'results/tuning/exp1_relu')\nplot_curves('Baseline (ReLU)', results, 'comparison_plots/exp1_relu.png')\nprint(f\"✅ Exp1: F1={results['test']['f1']:.4f}\")

## Experiment 2: Baseline (GELU)

In [ ]:
!python src/train.py --config config/tuning_experiments.yaml --experiment activation_gelu --save_dir results/tuning/exp2_gelu

In [ ]:
with open('results/tuning/exp2_gelu/results.json') as f:\n    results = json.load(f)\n\ncfg = {'activation':'gelu','receptors':128,'glomeruli':32,'lstm':256,'dropout':0.5,'batch':32,'sparse':0.001,'diverse':0.01}\nsave_to_drive('exp2_gelu', 'results/tuning/exp2_gelu')\nsave_results('exp2_gelu', cfg, results, 'results/tuning/exp2_gelu')\nplot_curves('Baseline (GELU)', results, 'comparison_plots/exp2_gelu.png')\nprint(f\"✅ Exp2: F1={results['test']['f1']:.4f}\")

## Experiment 3: More Receptors

In [ ]:
!python src/train.py --config config/hyperparameter_tuning.yaml --experiment exp3_more_receptors --save_dir results/tuning/exp3_rec256

In [ ]:
with open('results/tuning/exp3_rec256/results.json') as f:\n    results = json.load(f)\n\ncfg = {'activation':'gelu','receptors':256,'glomeruli':64,'lstm':256,'dropout':0.5,'batch':32,'sparse':0.001,'diverse':0.01}\nsave_to_drive('exp3_rec256', 'results/tuning/exp3_rec256')\nsave_results('exp3_rec256', cfg, results, 'results/tuning/exp3_rec256')\nplot_curves('More Receptors', results, 'comparison_plots/exp3_rec256.png')\nprint(f\"✅ Exp3: F1={results['test']['f1']:.4f}\")

## Experiment 4: More Glomeruli

In [ ]:
!python src/train.py --config config/hyperparameter_tuning.yaml --experiment exp4_more_glomeruli --save_dir results/tuning/exp4_glom64

In [ ]:
with open('results/tuning/exp4_glom64/results.json') as f:\n    results = json.load(f)\n\ncfg = {'activation':'gelu','receptors':128,'glomeruli':64,'lstm':256,'dropout':0.5,'batch':32,'sparse':0.001,'diverse':0.01}\nsave_to_drive('exp4_glom64', 'results/tuning/exp4_glom64')\nsave_results('exp4_glom64', cfg, results, 'results/tuning/exp4_glom64')\nplot_curves('More Glomeruli', results, 'comparison_plots/exp4_glom64.png')\nprint(f\"✅ Exp4: F1={results['test']['f1']:.4f}\")

## Experiment 5: Larger LSTM

In [ ]:
!python src/train.py --config config/hyperparameter_tuning.yaml --experiment exp5_larger_lstm --save_dir results/tuning/exp5_lstm512

In [ ]:
with open('results/tuning/exp5_lstm512/results.json') as f:\n    results = json.load(f)\n\ncfg = {'activation':'gelu','receptors':128,'glomeruli':32,'lstm':512,'dropout':0.5,'batch':32,'sparse':0.001,'diverse':0.01}\nsave_to_drive('exp5_lstm512', 'results/tuning/exp5_lstm512')\nsave_results('exp5_lstm512', cfg, results, 'results/tuning/exp5_lstm512')\nplot_curves('Larger LSTM', results, 'comparison_plots/exp5_lstm512.png')\nprint(f\"✅ Exp5: F1={results['test']['f1']:.4f}\")

## Experiment 6: Lower Dropout

In [ ]:
!python src/train.py --config config/hyperparameter_tuning.yaml --experiment exp6_lower_dropout --save_dir results/tuning/exp6_drop02

In [ ]:
with open('results/tuning/exp6_drop02/results.json') as f:\n    results = json.load(f)\n\ncfg = {'activation':'gelu','receptors':128,'glomeruli':32,'lstm':256,'dropout':0.2,'batch':32,'sparse':0.001,'diverse':0.01}\nsave_to_drive('exp6_drop02', 'results/tuning/exp6_drop02')\nsave_results('exp6_drop02', cfg, results, 'results/tuning/exp6_drop02')\nplot_curves('Lower Dropout', results, 'comparison_plots/exp6_drop02.png')\nprint(f\"✅ Exp6: F1={results['test']['f1']:.4f}\")

## Experiment 7: Larger Batch

In [ ]:
!python src/train.py --config config/hyperparameter_tuning.yaml --experiment exp7_larger_batch --save_dir results/tuning/exp7_batch64

In [ ]:
with open('results/tuning/exp7_batch64/results.json') as f:\n    results = json.load(f)\n\ncfg = {'activation':'gelu','receptors':128,'glomeruli':32,'lstm':256,'dropout':0.5,'batch':64,'sparse':0.001,'diverse':0.01}\nsave_to_drive('exp7_batch64', 'results/tuning/exp7_batch64')\nsave_results('exp7_batch64', cfg, results, 'results/tuning/exp7_batch64')\nplot_curves('Larger Batch', results, 'comparison_plots/exp7_batch64.png')\nprint(f\"✅ Exp7: F1={results['test']['f1']:.4f}\")

## Experiment 8: Strong Reg

In [ ]:
!python src/train.py --config config/hyperparameter_tuning.yaml --experiment exp8_strong_reg --save_dir results/tuning/exp8_strongreg

In [ ]:
with open('results/tuning/exp8_strongreg/results.json') as f:\n    results = json.load(f)\n\ncfg = {'activation':'gelu','receptors':128,'glomeruli':32,'lstm':256,'dropout':0.5,'batch':32,'sparse':0.1,'diverse':0.1}\nsave_to_drive('exp8_strongreg', 'results/tuning/exp8_strongreg')\nsave_results('exp8_strongreg', cfg, results, 'results/tuning/exp8_strongreg')\nplot_curves('Strong Reg', results, 'comparison_plots/exp8_strongreg.png')\nprint(f\"✅ Exp8: F1={results['test']['f1']:.4f}\")

---\n# Comparison\n---

In [ ]:
import pandas as pd\n\ndata = []\nfor e in all_experiments:\n    data.append({\n        'Name': e['name'],\n        'F1': f\"{e['results']['test']['f1']:.4f}\",\n        'Precision': f\"{e['results']['test']['precision']:.4f}\",\n        'Recall': f\"{e['results']['test']['recall']:.4f}\"\n    })\n\ndf = pd.DataFrame(data)\nprint(df.to_string(index=False))\ndf.to_csv('comparison_plots/results.csv', index=False)

## Push to GitHub

In [ ]:
!git config user.name \"Colab\"\n!git config user.email \"colab@exp.local\"\n!git add experiment_results/ comparison_plots/\n!git commit -m \"Hyperparameter tuning results\"\n!git push origin main\nprint('✅ Pushed to GitHub')